# 🫁 ChestNet on Google Colab: Part 1, the data pipeline (A → Z)

This notebook gets **all the data ready** for the four models, using the Kaggle **Random Sample of NIH Chest X-ray Dataset** (5,606 images). It uses your own `src/` code, so Colab and your laptop give exactly the same results.

```text
🔑 Secrets → 📁 Google Drive → 📦 Clone repo → ⬇️ Download the sample (5,606 X-rays)
   → ✂️ Patient-level splits → 📊 EDA → ✅ Pipeline check → ⚡ Warm cache → 👀 Visual check → 💾 Save outputs
```

**⏱️ Time:** about 10–15 min per session. The sample is small, so it's downloaded fresh every session, with no Drive backup needed.

## 🧰 One-time setup (before running anything)

1. **💻 Push the updated code.** `splits.py`, `eda.py` and `configs/base.yaml` were changed for the sample dataset. Commit them and push them to GitHub, because this notebook clones the repo from there.
2. **🔑 Get a Kaggle API key.** Go to kaggle.com → *Settings* → *API* → *Create New Token*. It downloads `kaggle.json` with your `username` and `key`.
3. **🗝️ Add Colab secrets.** Click the 🔑 icon in the left sidebar, add these, and switch on *Notebook access* for each:
   - `KAGGLE_USERNAME` and `KAGGLE_KEY` (from `kaggle.json`). If Kaggle gives you a single new-style token instead, add it as `KAGGLE_API_TOKEN`.
   - `GITHUB_TOKEN`: **only if the repo is private**. Use a GitHub *personal access token* with read access.
4. **⚡ Turn on the GPU.** *Runtime* → *Change runtime type* → **T4 GPU**.

Then run the cells **from top to bottom** ▶️

## 1️⃣ Check the GPU and disk space 🖥️

In [ ]:
import shutil, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("🎮 GPU:", gpu or "⚠️ NONE – go to Runtime > Change runtime type > T4 GPU")
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"💾 Free disk: {free_gb:.0f} GB  (the sample needs about 10 GB)")

## 2️⃣ Settings and Google Drive 📁
Google Drive keeps your data and results safe when Colab disconnects.

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/Deep-Learning-Models/ChestNet.git"
BRANCH    = "main"                                   # or your feature branch until it is merged
REPO_DIR  = "/content/ChestNet"                      # fast local disk (code + images live here)
DRIVE_DIR = "/content/drive/MyDrive/ChestNet_colab"  # permanent storage

from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

CACHE_TAR  = f"{DRIVE_DIR}/chestnet_cache.tar"       # resized 224 px tf.data cache

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

print("✅ Drive folder:", DRIVE_DIR)

## 3️⃣ Clone the ChestNet repo 📦

In [ ]:
token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} {url} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -3

assert "sample_labels.csv" in Path("configs/base.yaml").read_text() and \
       "parse_age" in Path("src/preprocessing/splits.py").read_text(), (
    "❌ This copy of the code is old. Push the updated splits.py, eda.py and configs/base.yaml "
    "to GitHub (or set BRANCH to the branch that has them) and run this cell again.")
print("✅ Code is up to date")

## 4️⃣ Install packages 🧩
Colab already has TensorFlow, Keras 3, pandas, scikit-learn and Matplotlib. Only `keras-hub` (for the ViT) is missing.

> ⚠️ Don't run `pip install -r requirements.txt` here. It includes `jupyter`, which can break Colab.

In [ ]:
!pip install -q "keras-hub>=0.18" "kaggle>=1.6"
import tensorflow as tf, keras, keras_hub, pandas as pd, sklearn
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| keras-hub", keras_hub.__version__)
print("GPUs seen by TensorFlow:", tf.config.list_physical_devices("GPU"))

## 5️⃣ Kaggle login 🔑

In [ ]:
api_token = secret("KAGGLE_API_TOKEN")
if api_token:
    os.environ["KAGGLE_API_TOKEN"] = api_token
else:
    os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME") or ""
    os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY") or ""
assert api_token or (os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_KEY"]), \
    "❌ Add KAGGLE_USERNAME + KAGGLE_KEY (or KAGGLE_API_TOKEN) in the 🔑 Secrets panel"
!kaggle datasets files nih-chest-xrays/sample | head -5
print("✅ Kaggle works")

## 6️⃣ Download the dataset ⬇️
Downloads the **Random Sample of NIH Chest X-ray Dataset** (5,606 X-rays). The label file goes to `data/sample_labels.csv` and the images to `data/raw/images/`.

In [ ]:
import zipfile
DL_DIR = Path("/content/nih_sample")
img_dir = Path("data/raw/images")
if len(list(img_dir.glob("*.png"))) < 5606:
    if not list(DL_DIR.rglob("*.png")):
        !kaggle datasets download nih-chest-xrays/sample -p {DL_DIR} --unzip
    for z in DL_DIR.rglob("*.zip"):                              # in case anything stayed zipped
        zipfile.ZipFile(z).extractall(z.parent); z.unlink()

    # Label file -> data/sample_labels.csv
    csv = next(DL_DIR.rglob("sample_labels.csv"))
    shutil.copy(csv, "data/sample_labels.csv")

    # Images -> data/raw/images/ (one copy per file name; the ZIP may contain duplicates)
    img_dir.mkdir(parents=True, exist_ok=True)
    for p in DL_DIR.rglob("*.png"):
        target = img_dir / p.name
        if not target.exists():
            shutil.move(str(p), target)
    shutil.rmtree(DL_DIR)

labels = pd.read_csv("data/sample_labels.csv")
n_img = len(list(img_dir.glob("*.png")))
print(f"✅ {len(labels)} labels, {n_img} images, {labels['Patient ID'].nunique()} patients")
if n_img != len(labels):
    print("⚠️ Image and label counts differ – the split step will name any missing image")
print("\nImages per label (an image can have several):")
print(labels["Finding Labels"].str.split("|").explode().value_counts().to_string())

## 7️⃣ Restore the image cache from Drive ♻️
From the second session on, this restores the resized 224 px images, so training starts fast.

In [ ]:
if os.path.exists(CACHE_TAR):
    !tar -xf "{CACHE_TAR}" -C {REPO_DIR}
    print("✅ tf.data cache restored")
else:
    print("🆕 No cache yet – Step 11 will build it")

## 8️⃣ Patient-level train / val / test split ✂️
Runs your normal split script: 70 / 15 / 15 **by patient**, keeping the most label-balanced of 200 candidate splits. All images of the 7 diseases are used, and *No Finding* is capped at 1,500.

> ⚠️ **Check the Pneumonia row.** The sample has only a few dozen Pneumonia images, so val and test get about 10 each. Its test metrics will be uncertain: mention this as a limitation in the report.

In [ ]:
!python -m src.preprocessing.splits
import json
display(pd.read_csv("data/splits/summary.csv", index_col=0))
print(json.dumps(json.load(open("data/splits/split_report.json")), indent=2))

## 9️⃣ Exploratory data analysis 📊
Creates report-ready figures in `results/eda/`.

In [ ]:
!python -m src.eda
from IPython.display import Image as IPImage, display, Markdown
for png in sorted(Path("results/eda").glob("*.png")):
    display(Markdown(f"**{png.name}**")); display(IPImage(str(png), width=650))

## 🔟 Verify the shared training rules ✅
Checks for leakage, input shapes and pixel ranges, and confirms augmentation only touches the training set. Gives you `results/protocol/` for Report Section 5.

In [ ]:
!python -m src.check_pipeline
display(Markdown(Path("results/protocol/experimental_protocol.md").read_text()))
display(IPImage("results/protocol/augmentation_preview.png", width=650))

## 1️⃣1️⃣ Warm up the tf.data cache ⚡
Decoding 1024 px PNGs is slow. The pipeline caches the 224 px images after the first full pass.

**Why do this now?** `train.py` records *time per epoch*. If the cache is cold, the first model you train gets a slow first epoch, which makes the cost comparison unfair. After this step, every model starts with the same warm cache. The cache is keyed only on the image list and image size, so all four models (any batch size) share it.

In [ ]:
import time
from src.config import load_config
from src.preprocessing.splits import load_splits
from src.preprocessing.dataset import make_dataset

cfg = load_config("configs/base.yaml")
train_df, val_df, test_df = load_splits(cfg)
# Same calls as train.py (train / val) and evaluate.py (test) -> same cache files.
for name, df, kw in [("train", train_df, dict(training=True, split="train")),
                     ("val",   val_df,   dict(training=False, split="val")),
                     ("test",  test_df,  dict(training=False))]:
    t = time.perf_counter()
    n = sum(int(x.shape[0]) for x, _ in make_dataset(df, cfg, **kw))
    print(f"⚡ {name:5s}: {n} images cached in {time.perf_counter()-t:.0f} s")

!tar -cf "{CACHE_TAR}" data/processed/cache
print("💾 Cache saved to Drive")

## 1️⃣2️⃣ Look at one training batch 👀
A last check by eye: X-rays look right, the augmentation is mild, and the labels make sense.

In [ ]:
import matplotlib.pyplot as plt
classes = cfg["data"]["classes"]
x, y = next(iter(make_dataset(train_df, cfg, training=True, split="train")))
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, img, lab in zip(axes.flat, x.numpy(), y.numpy()):
    ax.imshow(img[..., 0], cmap="gray", vmin=0, vmax=255); ax.axis("off")
    ax.set_title(", ".join(c for c, v in zip(classes, lab) if v) , fontsize=9)
plt.suptitle(f"Training batch {tuple(x.shape)}  pixels {x.numpy().min():.0f}–{x.numpy().max():.0f}")
plt.tight_layout(); plt.show()

## 1️⃣3️⃣ Save the outputs 💾
Copies the splits and report files to Drive, and gives you a ZIP to commit to GitHub.

In [ ]:
for folder in ["data/splits", "results"]:
    shutil.copytree(folder, f"{DRIVE_DIR}/{folder}", dirs_exist_ok=True)
shutil.make_archive("/content/chestnet_data_outputs", "zip", REPO_DIR, "data/splits")
shutil.make_archive("/content/chestnet_report_outputs", "zip", REPO_DIR, "results")
print("✅ Copied to", DRIVE_DIR)

from google.colab import files
files.download("/content/chestnet_data_outputs.zip")
files.download("/content/chestnet_report_outputs.zip")

## 🎉 The data pipeline is done

| ✅ You now have | 📍 Where |
|---|---|
| 5,606 X-rays + `sample_labels.csv` | Colab disk (downloaded again each session) |
| Patient-level splits and leakage report | `data/splits/` |
| EDA figures and `eda_summary.json` | `results/eda/` |
| Protocol table, class weights, augmentation preview | `results/protocol/` |
| Warm 224 px cache | `data/processed/cache/` (backup on Drive) |

### 👉 What to do next
1. **📤 Commit to GitHub** `data/splits/*.csv`, `split_report.json`, `results/eda/` and `results/protocol/`. Then the whole team trains on the **same split**. The `path` column is relative (`data/raw/images/...`), so it works on Colab and on Windows.
2. **🏋️ Train.** In a new session, run Steps 1–7 (about 5–10 min), then:
   `!python -m src.train --config configs/custom_cnn.yaml --epochs 2` as a quick test, then the full runs.
   Copy `results/` and `saved_models/` to Drive after each model.